# Communicative agents with OPC integration and MaintenanceAgents

This notebook builds on the code from **Communcative_agents_with_OPC_Hot_Monitor.ipynb**
and shows how to further integrate an external OPC UA client module:

`OPC_Agent_Client.py`

The goal is to:

- to show the intergration of a more OPC UA communication
- to enhance interaction between agents, i.e. MachineAgents and MaintennaceAgents

`OPC_Agent_Client.py` is responsible for:

- connecting to the factory OPC UA server,
- handling machine temperatures
- handling repair jobs


## 1. High–level architecture

The integration follows a **loose coupling** approach:

- The **Mesa model** simulates agents and their environment (e.g. factories, machines,
  maintenance teams, etc.).
- The **OPC Agent client** runs as a separate asynchronous task,
  talking to the real (or simulated) OPC UA server.
- Both live in the **same Jupyter process**, so they can be started from the same notebook,
  but they do not block each other.

This is intentionally simple and suitable for teaching:

- You can run and inspect the Mesa model as before.
- You can start or stop the OPC UA HOT monitor independently.
- This implementation shows, how to tighten the coupling (i.e. agents reacting to `*_RepairNeeded` flags
  or calling OPC UA methods).

## 2. Environment setup – Mesa 2.x and OPC UA client

In [ ]:
import asyncio

# The OPC Agent client exported earlier from the dedicated notebook.
# It must be located in the same directory as this notebook, as:
#     OPC_Agent_Client.py
import OPC_Agent_Client as agent_client

import math
from datetime import datetime

from asyncua import ua, Client

# Configuration – these values must match your server notebook
SERVER_URL = "opc.tcp://localhost:4840/freeopcua/server/"
FACTORY_NS_URI = "http://ostfalia.de/ipt/factory"

print("Event loop setup complete. Server endpoint:", SERVER_URL)

## 3. MESA Agents from Reactive Agents

The following cells are taken directly from the original
*_agents_with_OPC_Hot_Monitor* notebook. They define the advanced Mesa model,
agents, and visualisation.

**Here we extend:**
1. **MachineAgent**
   - gets a new member opc_name (see: __init__(self, unique_id, model, threshold=70, opc_name:str=None):)
2. **FactoryModelExtended**
   - adds opc_name while creating MachineAgents
   - starts OPC publisher with this instance
   
3. **MaintenanceAgent**
    - Behaviour each step:
        1. If it has no current target, pick a machine with repair_jobs[name] == True
           that is not already targeted by another MaintenanceAgent.
        2. Move one step towards the target machine's grid position.
        3. If it arrives on that cell, set the machine's temperature to 20.0 °C,
           clear the busy flag and mark the job as done in model.repair_jobs,
           and clear the RepairNeeded flag on the OPC UA server.


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Extended reactive machine agent with multiple temperature states.

    States:
        COOL   : temperature < 0.75 * threshold
        WARM   : 0.75 * threshold <= temperature <= threshold
        HOT    : temperature > threshold
        OK     : fallback state
    """

    def __init__(self, unique_id, model, threshold=70, opc_name:str=None):
        super().__init__(unique_id, model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False  # used by OrderAgent
        ## NEW ##
        self.opc_name = opc_name  # name of the corresponding OPC UA machine (e.g. "M01")
        # Clear RepairNeeded flag on the OPC UA server via the HOT monitor client
        try:
            agent_client.clear_repair_flag_sync(self.opc_name)
        except Exception as exc:
            print(f"[MAINT] Failed to clear OPC UA flag for {self.opc_name}: {exc}")
        ## END NEW ##

    def sense_temperature(self):
        """Simple sensor model: temperature + random noise.
        Machines that are not HOT slowly heat up.
        """
        if self.state != "HOT":
            noise = random.uniform(-1, 2)
            self.temperature = self.temperature + noise

    def decide(self):
        """Reactive decision rule based only on the current temperature."""
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        pass

    def step(self):
        self.sense_temperature()
        self.decide()
        self.act()

## NEW ##
class MaintenanceAgent(Agent):
    """Agent that moves around the grid and repairs machines based on OPC UA jobs.

    Behaviour each step:
        1. If it has no current target, pick a machine with repair_jobs[name] == True
           that is not already targeted by another MaintenanceAgent.
        2. Move one step towards the target machine's grid position.
        3. If it arrives on that cell, set the machine's temperature to 20.0 °C,
           clear the busy flag and mark the job as done in model.repair_jobs,
           and clear the RepairNeeded flag on the OPC UA server.
    """

    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.target_machine_name = None

    # ---- helpers for job selection and movement ---------------------------------

    def get_machine_position(self, machine_name):
        """Return the grid position of the machine with given opc_name, or None."""
        for agent in self.model.schedule.agents:
            if isinstance(agent, MachineAgent) and agent.opc_name == machine_name:
                return agent.pos
        return None

    def distance_to_machine(self, machine_name):
        """Simple Manhattan distance to a given machine."""
        pos = self.get_machine_position(machine_name)
        if pos is None or self.pos is None:
            return float("inf")
        x, y = self.pos
        tx, ty = pos
        return abs(tx - x) + abs(ty - y)

    def move_towards(self, target_pos):
        """Move one step in the Moore neighbourhood towards target_pos."""
        if target_pos is None or self.pos is None:
            return

        x, y = self.pos
        tx, ty = target_pos

        dx = 0
        dy = 0
        if tx > x:
            dx = 1
        elif tx < x:
            dx = -1
        if ty > y:
            dy = 1
        elif ty < y:
            dy = -1

        new_pos = (x + dx, y + dy)
        # ensure new_pos is within bounds
        if 0 <= new_pos[0] < self.model.width and 0 <= new_pos[1] < self.model.height:
            self.model.grid.move_agent(self, new_pos)

    # ---- job selection & repair --------------------------------------------------

    def choose_target(self):
        """Pick an available machine with a pending repair job as target (if any)."""
        jobs = getattr(self.model, "repair_jobs", {})
        if not jobs:
            return

        # all machines with pending jobs
        pending = [name for name, flag in jobs.items() if flag]
        if not pending:
            return

        # jobs that are already targeted by other maintenance agents
        taken = {
            a.target_machine_name
            for a in self.model.schedule.agents
            if isinstance(a, MaintenanceAgent) and a.target_machine_name is not None
        }

        # remaining jobs that no one else is already following
        available = [name for name in pending if name not in taken]
        if not available:
            return

        # choose the closest available machine
        self.target_machine_name = min(
            available, key=lambda name: self.distance_to_machine(name)
        )

    def repair_if_on_target(self):
        """If on the same cell as the target machine, perform the repair."""
        if self.target_machine_name is None:
            return

        cell_mates = self.model.grid.get_cell_list_contents([self.pos])
        for agent in cell_mates:
            if isinstance(agent, MachineAgent) and agent.opc_name == self.target_machine_name:
                # 1) repair in the Mesa model
                agent.temperature = 20.0
                agent.state = "OK"
                agent.busy = False

                # 2) clear local repair flag in the model
                if hasattr(self.model, "repair_jobs"):
                    self.model.repair_jobs[self.target_machine_name] = False

                # 3) clear RepairNeeded flag on the OPC UA server via the agent client
                try:
                    agent_client.clear_repair_flag_sync(self.target_machine_name)
                except Exception as exc:
                    print(f"[MAINT] Failed to clear OPC UA flag for {self.target_machine_name}: {exc}")

                print(f"[MAINT] Repaired {self.target_machine_name} at position {self.pos}")
                self.target_machine_name = None
                break

    def step(self):
        # If our current target is no longer a job (e.g. cleared by server or other agent), drop it
        jobs = getattr(self.model, "repair_jobs", {})
        if self.target_machine_name is not None and not jobs.get(self.target_machine_name, False):
            self.target_machine_name = None

        # 1. If we do not have a target, try to choose one based on OPC UA jobs
        if self.target_machine_name is None:
            self.choose_target()

        # 2. If we have a target, move towards it
        if self.target_machine_name is not None:
            target_pos = self.get_machine_position(self.target_machine_name)
            if target_pos is not None and target_pos != self.pos:
                self.move_towards(target_pos)

        # 3. Try to repair if we are on top of the target machine
        self.repair_if_on_target()
## END NEW ##

class OrderAgent(Agent):
    """Agent representing a simple production order.

    Behaviour each step:
        1. Move randomly through the grid.
        2. Search for an available (non-HOT, not busy) MachineAgent.
        3. When a machine is found, mark it as busy and remove the order.
    """

    def step(self):
        # 1. Move randomly
        neighbours = self.model.grid.get_neighborhood(self.pos, moore=True, include_center=True)
        new_pos = random.choice(neighbours)
        self.model.grid.move_agent(self, new_pos)

        # 2. Check for available machine in the new cell
        cell_mates = self.model.grid.get_cell_list_contents([self.pos])
        for agent in cell_mates:
            if isinstance(agent, MachineAgent) and agent.state != "HOT" and not agent.busy:
                # 3. Assign order to this machine
                agent.busy = True
                # optional: slightly increase temperature to simulate load
                agent.temperature += 5
                # Remove this order from the model
                self.model.grid.remove_agent(self)
                self.model.schedule.remove(self)
                break


from mesa import Model
from mesa.time import RandomActivation
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.schedule.agents if isinstance(a, MachineAgent) and a.state == "HOT")


def count_warm_machines(model):
    return sum(1 for a in model.schedule.agents if isinstance(a, MachineAgent) and a.state == "WARM")


def count_orders(model):
    return sum(1 for a in model.schedule.agents if isinstance(a, OrderAgent))


class FactoryModelExtended(Model):
    """Extended factory model with machines, maintenance agents and order agents.

    This version adds:
    - a dictionary `repair_jobs` mapping machine opc_name -> bool,
    - a call to `sync_repair_jobs_from_opcua(self)` in each step, so that
      MaintenanceAgents can react to external OPC UA job flags.
    """

    def __init__(self, width=10, height=10, threshold=70,
                 n_maintenance=2, n_orders=5, n_machines=10):
        super().__init__()
        self.width = width
        self.height = height
        self.n_machines = n_machines
        self.threshold = threshold
        self.n_maintenance = n_maintenance
        self.n_orders = n_orders

        self.schedule = RandomActivation(self)
        self.grid = MultiGrid(width, height, torus=False)
        
        ## NEW ##
        # dictionary for OPC UA repair jobs
        self.repair_jobs = {}
        ## END NEW ##
        
        # 1. Create machine agents
        agent_id = 0
        for _ in range(self.n_machines):
            x = self.random.randrange(self.width)
            y = self.random.randrange(self.height)
            opc_name = f"M{agent_id+1:02d}"
            agent = MachineAgent(agent_id, self, threshold=self.threshold, opc_name=opc_name)
            self.schedule.add(agent)
            self.grid.place_agent(agent, (x, y))
            agent_id += 1

        # 2. Create maintenance agents
        for _ in range(self.n_maintenance):
            x = self.random.randrange(self.width)
            y = self.random.randrange(self.height)
            maint = MaintenanceAgent(agent_id, self)
            self.schedule.add(maint)
            self.grid.place_agent(maint, (x, y))
            agent_id += 1

        # 3. Create order agents
        for _ in range(self.n_orders):
            x = self.random.randrange(self.width)
            y = self.random.randrange(self.height)
            order = OrderAgent(agent_id, self)
            self.schedule.add(order)
            self.grid.place_agent(order, (x, y))
            agent_id += 1

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines": count_hot_machines,
                "WarmMachines": count_warm_machines,
                "Orders": count_orders,
            }
        )
        
        
        # start OPC publisher with this instance
        start_temperature_publisher(self, interval=1.0)
        
        
    def step(self):
        ## NEW ##
        # 1. Pull current repair job flags from OPC UA into the model
        sync_repair_jobs_from_opcua(self)
        ## END NEW ##
        # 2. Normal Mesa update
        self.datacollector.collect(self)
        self.schedule.step()


### Visualization for the extended model
Nothing changes here...

In [ ]:
from mesa.visualization.modules import CanvasGrid, ChartModule
from mesa.visualization.ModularVisualization import ModularServer


def extended_portrayal(agent):
    if agent is None:
        return

    # Machine agents: coloured rectangles
    if isinstance(agent, MachineAgent):
        text_color = "white"
        if agent.state == "COOL":
            color = "blue"
        elif agent.state == "HOT":
            color = "red"
        elif agent.state == "WARM":
            color = "yellow"
            text_color = "black"
        else:
            color = "green"

        return {
            "Shape": "rect",
            "w": 0.8,
            "h": 0.8,
            "Filled": "true",
            "Layer": 0,
            "Color": color,
            "text": str(round(agent.temperature, 1)),
            "text_color": text_color,
        }

    # Maintenance agents: blue circles
    if isinstance(agent, MaintenanceAgent):
        return {
            "Shape": "circle",
            "r": 0.4,
            "Filled": "true",
            "Layer": 2,
            "Color": "cyan",
        }

    # Order agents: black triangles
    if isinstance(agent, OrderAgent):
        return {
            "Shape": "rect",
            "w": 0.5,
            "h": 0.5,
            "Filled": "true",
            "Layer": 1,
            "Color": "black",
            "text": str(round(agent.unique_id, 1)),
            "text_color": "white",
        }


width, height = 10, 10
grid = CanvasGrid(extended_portrayal, width, height, 500, 500)

chart = ChartModule(
    [
        {"Label": "HotMachines", "Color": "Red"},
        {"Label": "WarmMachines", "Color": "Yellow"},
        {"Label": "Orders", "Color": "Black"},
    ],
    data_collector_name="datacollector",
)

## 4. Integrating the OPC UA HOT monitor client

The HOT monitor client (from `OPC_Agent_Client.py`) exposes, among others,
two key coroutine functions:

- `polling_hot_monitor(runtime_seconds: float, poll_interval: float)`  
- `subscription_hot_monitor(runtime_seconds: float, publishing_interval_ms: int)`

Both functions:

- connect to the OPC UA factory server,
- discover all machines and the corresponding `Mxx_RepairNeeded` job flags,
- watch temperatures and set `Mxx_RepairNeeded = True` whenever a machine becomes HOT.

In this notebook, we simply **run the HOT monitor in parallel** with the Mesa model.
For Jupyter this is convenient: we can schedule the monitor as a background task
while the Mesa simulation runs in the foreground.


### 4.1 Helper: start the HOT monitor as a background task

The helper below starts the HOT monitor as an `asyncio` task. This works well in
Jupyter when an event loop is already running (for example because `IPython`
manages one internally).

In [ ]:
hot_monitor_task = None  # global handle so we can cancel if needed

def start_hot_monitor_in_background(
    mode: str = "subscription",
    runtime_seconds: float = 120.0,
    interval: float = 1.0,
    publishing_interval_ms: int = 500,
):
    """Start the OPC UA HOT monitor as a background asyncio task.

    Parameters
    ----------
    mode:
        Either "subscription" (default) or "polling".
    runtime_seconds:
        How long the monitor should run before it stops itself.
    interval:
        Polling interval in seconds (used only in polling mode).
    publishing_interval_ms:
        Subscription publishing interval in milliseconds (used only in subscription mode).
    """
    global hot_monitor_task

    loop = asyncio.get_event_loop()

    if mode == "polling":
        coro = agent_client.polling_hot_monitor(
            runtime_seconds=runtime_seconds,
            poll_interval=interval,
        )
    else:
        # default: subscription mode
        coro = agent_client.subscription_hot_monitor(
            runtime_seconds=runtime_seconds,
            publishing_interval_ms=publishing_interval_ms,
        )

    hot_monitor_task = loop.create_task(coro)
    print(f"Started HOT monitor in background ({mode} mode). Task:", hot_monitor_task)


def cancel_hot_monitor():
    """Cancel the background HOT monitor task, if running."""
    global hot_monitor_task
    if hot_monitor_task is not None and not hot_monitor_task.done():
        hot_monitor_task.cancel()
        print("HOT monitor task cancelled.")
    else:
        print("No running HOT monitor task found.")

## 5. Implementimg the temperature publisher

Persistent OPC UA temperature publisher using asyncua.

    - Connects once to the OPC UA server.
    - Uses agent_client.discover_machines_with_jobs(...) to find Temperature nodes.
    - In a loop, iterates over all MachineAgent instances in the Mesa model
      and writes their temperatures to the corresponding OPC UA nodes.


In [ ]:
temperature_publisher_task = None  # global handle for the background task


async def persistent_opcua_temperature_publisher(model, interval: float = 1.0):
    """
    Persistent OPC UA temperature publisher using asyncua.

    - Connects once to the OPC UA server.
    - Uses agent_client.discover_machines_with_jobs(...) to find Temperature nodes.
    - In a loop, iterates over all MachineAgent instances in the Mesa model
      and writes their temperatures to the corresponding OPC UA nodes.
    """
    print("[PUBLISHER] Trying to connect to OPC UA server... (model id", id(model), ")")
    async with Client(url=agent_client.SERVER_URL) as client:
        print("[PUBLISHER] Connected to OPC UA server:", agent_client.SERVER_URL, "for model id", id(model))

        # Discover machine nodes once
        machines_nodes = await agent_client.discover_machines_with_jobs(client)
        # machines_nodes: dict "M01" -> MachineNodes(temp_node=..., job_node=...)

        try:
            while True:
                for agent in model.schedule.agents:
                    if not isinstance(agent, MachineAgent):
                        continue
                    if agent.opc_name is None:
                        continue

                    machine_name = agent.opc_name
                    nodes = machines_nodes.get(machine_name)
                    if nodes is None:
                        # No corresponding Temperature node on the OPC UA server
                        continue

                    temp_node = nodes.temp_node
                    value = float(agent.temperature)

                    try:
                        await temp_node.write_value(ua.Variant(value, ua.VariantType.Double))
                        print(f"[PUBLISHER] Temperature writen {machine_name} temperature {value} °C")
                    except Exception as exc:
                        print(f"[PUBLISHER] Failed to write {machine_name} temperature ({exc})")

                await asyncio.sleep(interval)

        except asyncio.CancelledError:
            print("[PUBLISHER] Temperature publisher task cancelled.")
            raise


def start_temperature_publisher(model, interval: float = 1.0):
    """
    Start the persistent OPC UA temperature publisher as an asyncio background task.

    The task will:
    - connect once to the OPC UA server,
    - periodically write all MachineAgent temperatures.
    """
    global temperature_publisher_task
    loop = asyncio.get_event_loop()

    # Falls noch ein alter Task läuft: erst abbrechen
    if temperature_publisher_task is not None and not temperature_publisher_task.done():
        print("[PUBLISHER] Cancelling previous temperature publisher task...")
        temperature_publisher_task.cancel()

    temperature_publisher_task = loop.create_task(
        persistent_opcua_temperature_publisher(model, interval)
    )
    print("[PUBLISHER] Background task started:", temperature_publisher_task)


def cancel_temperature_publisher():
    """Cancel the background OPC UA temperature publisher, if running."""
    global temperature_publisher_task
    if temperature_publisher_task is not None and not temperature_publisher_task.done():
        temperature_publisher_task.cancel()
        print("[PUBLISHER] Background task cancelled.")
    else:
        print("[PUBLISHER] No running background task found.")


## 5.1. Read repair jobs from server

In [ ]:
def sync_repair_jobs_from_opcua(model):
    # Ensure dict exists
    if not hasattr(model, "repair_jobs"):
        model.repair_jobs = {}

    model.repair_jobs = agent_client.read_all_repair_flags_sync()

### 6 Example: run Mesa model and HOT monitor together, publish temperatures in the background

A typical workflow in this notebook could be:

1. Run the cells from **Section 3** above to define the Mesa model and agents.  
2. Start the OPC UA HOT monitor in the background:
   ```python
   start_hot_monitor_in_background(
       mode="subscription",
       runtime_seconds=300.0,
       publishing_interval_ms=500,
   )
   ```
3. Run your Mesa simulation as usual (batch run, `model.step()` loop, or visualisation).  
4. Optionally, cancel the HOT monitor earlier:
   ```python
   cancel_hot_monitor()
   ```

Below is a small helper cell that shows how you might run a simple Mesa
simulation loop while the HOT monitor is active.


In [ ]:
# Example skeleton – adapt this to your concrete Mesa model / class names.
# This cell assumes that Section 7 defined a model class named `FactoryModel`
# (change the class name if your model is called differently).

def run_simulation():
    import nest_asyncio
    nest_asyncio.apply()

    server_ext = ModularServer(
        FactoryModelExtended,
        [grid, chart],
        "Extended Factory Model with Maintenance and Orders",
        {"width": width, "height": height, "threshold": 60,
         "n_maintenance": 2, "n_orders": 50, "n_machines":10},
    )
    server_ext.port = 8523
    server_ext.launch()
    
# Example usage (once the model is defined):
start_hot_monitor_in_background(mode="subscription", runtime_seconds=1000)
run_simulation()

## 7. Communicative maintenance agents with OPC UA

In this extended version of the model, the *MaintenanceAgent* is no longer purely
reactive to the local machine state (`state == "HOT"`). Instead, it reacts to
information coming from the **external OPC UA server**.

The idea is to demonstrate how agents can become *communicative* / *cooperative*
by using a shared infrastructure (here: OPC UA) rather than only local sensing.

### Task description

1. The HOT monitor client (`OPC_Hot_Monitor_Client.py`) observes machine
   temperatures on the OPC UA server and sets boolean job flags:
   `Factory/Maintenance/Jobs/Mxx_RepairNeeded`.

2. Extend the **MaintenanceAgent** so that it:

   * periodically receives the information which machines are marked as
     `RepairNeeded` (via a mapping in the Mesa model),
   * selects one of these machines as its *target*,
   * moves step by step towards the grid cell of that machine,
   * once it reaches the target cell, it performs a **repair**:
     - set the machine's `temperature` to `20.0` °C,
     - reset the local repair flag for that machine.

3. Extend the **FactoryModelExtended** to maintain a simple dictionary
   `repair_jobs`, mapping OPC machine names (e.g. `"M01"`, `"M02"`, …) to
   boolean values indicating whether a repair job is pending.

4. Provide a function that **pulls** the `Mxx_RepairNeeded` flags from the
   OPC UA server into this dictionary, so that the MaintenanceAgents can
   base their behaviour on external information.